In [1]:
import pandas as pd
import os, sys
os.getcwd()

'C:\\Users\\bhand\\Documents\\Codes\\smart-retail-forecasting\\notebooks'

In [2]:
os.chdir(os.path.dirname(os.getcwd()))

In [80]:
def load_products():

    return pd.read_json(
        "data/raw/products/products.json"
    )

df = load_products()

In [81]:
df.columns = (
        df.columns
        .str.lower()
        .str.strip()
    )

In [82]:
def make_hashable(x):
    if isinstance(x, list):
        return tuple(make_hashable(i) for i in x)   # recurse into list
    elif isinstance(x, dict):
        return tuple(sorted((k, make_hashable(v)) for k, v in x.items()))  # recurse into dict
    elif isinstance(x, set):
        return frozenset(make_hashable(i) for i in x)  # recurse into set
    elif isinstance(x, bytearray):
        return bytes(x)
    else:
        return x

# Apply across the whole DataFrame
df = df.apply(lambda col: col.map(make_hashable))

In [83]:
df['reviews'][1]

((('comment', 'Great product!'),
  ('date', '2025-04-30T09:41:02.053Z'),
  ('rating', 5),
  ('reviewerEmail', 'savannah.gomez@x.dummyjson.com'),
  ('reviewerName', 'Savannah Gomez')),
 (('comment', 'Awesome product!'),
  ('date', '2025-04-30T09:41:02.053Z'),
  ('rating', 4),
  ('reviewerEmail', 'christian.perez@x.dummyjson.com'),
  ('reviewerName', 'Christian Perez')),
 (('comment', 'Poor quality!'),
  ('date', '2025-04-30T09:41:02.053Z'),
  ('rating', 1),
  ('reviewerEmail', 'nicholas.bailey@x.dummyjson.com'),
  ('reviewerName', 'Nicholas Bailey')))

In [84]:
df.drop_duplicates(inplace=True)

In [85]:
df.head()

,id,title,description,category,price,discountpercentage,rating,stock,tags,brand,...,dimensions,warrantyinformation,shippinginformation,availabilitystatus,reviews,returnpolicy,minimumorderquantity,meta,images,thumbnail
0,1,Essence Mascara Lash Princess,The Essence Mascara Lash Princess is a popular...,beauty,9.99,10.48,2.56,99,"(beauty, mascara)",Essence,...,"((depth, 22.99), (height, 13.08), (width, 15.14))",1 week warranty,Ships in 3-5 business days,In Stock,"(((comment, Would not recommend!), (date, 2025...",No return policy,48,"((barcode, 5784719087687), (createdAt, 2025-04...",(https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...
1,2,Eyeshadow Palette with Mirror,The Eyeshadow Palette with Mirror offers a ver...,beauty,19.99,18.19,2.86,34,"(beauty, eyeshadow)",Glamour Beauty,...,"((depth, 27.67), (height, 22.47), (width, 9.26))",1 year warranty,Ships in 2 weeks,In Stock,"(((comment, Great product!), (date, 2025-04-30...",7 days return policy,20,"((barcode, 9170275171413), (createdAt, 2025-04...",(https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...
2,3,Powder Canister,The Powder Canister is a finely milled setting...,beauty,14.99,9.84,4.64,89,"(beauty, face powder)",Velvet Touch,...,"((depth, 20.59), (height, 27.93), (width, 29.27))",3 months warranty,Ships in 1-2 business days,In Stock,"(((comment, Would buy again!), (date, 2025-04-...",No return policy,22,"((barcode, 8418883906837), (createdAt, 2025-04...",(https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...
3,4,Red Lipstick,The Red Lipstick is a classic and bold choice ...,beauty,12.99,12.16,4.36,91,"(beauty, lipstick)",Chic Cosmetics,...,"((depth, 22.17), (height, 28.38), (width, 18.11))",3 year warranty,Ships in 1 week,In Stock,"(((comment, Great product!), (date, 2025-04-30...",7 days return policy,40,"((barcode, 9467746727219), (createdAt, 2025-04...",(https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...
4,5,Red Nail Polish,The Red Nail Polish offers a rich and glossy r...,beauty,8.99,11.44,4.32,79,"(beauty, nail polish)",Nail Couture,...,"((depth, 29.84), (height, 16.48), (width, 21.63))",1 month warranty,Ships overnight,In Stock,"(((comment, Poor quality!), (date, 2025-04-30T...",No return policy,22,"((barcode, 4063010628104), (createdAt, 2025-04...",(https://cdn.dummyjson.com/product-images/beau...,https://cdn.dummyjson.com/product-images/beaut...


In [86]:
numeric_cols = df.select_dtypes(
        include="number"
    ).columns
numeric_cols

Index(['id', 'price', 'discountpercentage', 'rating', 'stock', 'weight',
       'minimumorderquantity'],
      dtype='str')

In [87]:
missing_count = df[numeric_cols].isna().sum().sum()
print("Total changes:", missing_count)

Total changes: 0


In [88]:
def remove_price_outliers(df):

    q1 = df["price"].quantile(0.25)
    q3 = df["price"].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    # Filtered DataFrame
    filtered_df = df[(df["price"] >= lower) & (df["price"] <= upper)]

    # Count changes
    removed_count = len(df) - len(filtered_df)
    removed_pct = removed_count / len(df) * 100

    print(f"Removed {removed_count} rows ({removed_pct:.2f}% of data)")

    return filtered_df
    

In [89]:
df = remove_price_outliers(df)

Removed 5 rows (16.67% of data)


In [90]:
df["price_bucket"] = pd.qcut(df["price"], q=4, labels=["low","mid-low","mid-high","high"])

In [91]:
df["discount_percent"] = (
    df["discountpercentage"]
)

In [92]:
df["inventory_value"] = (
    df["price"] * df["stock"]
)

In [93]:
def process_holidays(df):

    df["date"] = pd.to_datetime(
        df["date"]
    )

    df["month"] = df["date"].dt.month

    df["day"] = df["date"].dt.day

    return df

In [95]:
def tuple_to_dict(x):
    # If it's a tuple of pairs, convert to dict
    if isinstance(x, tuple) and all(isinstance(i, tuple) and len(i) == 2 for i in x):
        return {k: tuple_to_dict(v) for k, v in x}
    # If it's a list or tuple of mixed items, recurse
    elif isinstance(x, (list, tuple)):
        return [tuple_to_dict(i) for i in x]
    else:
        return x

In [96]:
meta_dict = {k: tuple_to_dict(v) for k, v in dict(df.meta).items()}

meta_dict

In [139]:
def load_holidays():

    return pd.read_json(
        "data/raw/holidays/holidays.json"
    )
df = load_holidays()

In [141]:
meta_cell = df.loc[df["response"].notna(), "response"].iloc[0]
df = pd.DataFrame(meta_cell)

In [113]:
df.columns = (
        df.columns
        .str.lower()
        .str.strip()
    )

In [115]:
def make_hashable(x):
        if isinstance(x, list):
            return tuple(make_hashable(i) for i in x)   # recurse into list
        elif isinstance(x, dict):
            return tuple(sorted((k, make_hashable(v)) for k, v in x.items()))  # recurse into dict
        elif isinstance(x, set):
            return frozenset(make_hashable(i) for i in x)  # recurse into set
        elif isinstance(x, bytearray):
            return bytes(x)
        else:
            return x

# Apply across the whole DataFrame
df = df.apply(lambda col: col.map(make_hashable))

In [117]:
df.drop_duplicates(inplace=True)

In [108]:
def handle_missing_values(df):

    numeric_cols = df.select_dtypes(
        include="number"
    ).columns

    df[numeric_cols] = (
        df[numeric_cols]
        .fillna(
            df[numeric_cols].median()
        )
    )

df = handle_missing_values(df)

In [165]:
df["date_parsed"] = pd.to_datetime(df["date"].apply(lambda x: x['iso'] if pd.notna(x) else None), format='ISO8601', utc=True)

In [168]:
df.head(20)

,name,description,country,date,type,primary_type,canonical_url,urlid,locations,states,date_parsed
0,New Year's Day,New Year's Day is celebrated many countries su...,"{'id': 'in', 'name': 'India'}","{'iso': '2026-01-01', 'datetime': {'year': 202...",[Optional holiday],Restricted Holiday,https://calendarific.com/holiday/india/new-yea...,india/new-year-day,All,All,2026-01-01 00:00:00+00:00
1,Hazarat Ali's Birthday,Hazarat Ali's Birthday is a restricted holiday...,"{'id': 'in', 'name': 'India'}","{'iso': '2026-01-03', 'datetime': {'year': 202...",[Optional holiday],Restricted Holiday,https://calendarific.com/holiday/india/harazar...,india/harazart-ali-birthday,All,All,2026-01-03 00:00:00+00:00
2,Pongal,"Many southern states in India, particularly Ta...","{'id': 'in', 'name': 'India'}","{'iso': '2026-01-14', 'datetime': {'year': 202...","[Hinduism, Optional holiday]",Restricted Holiday,https://calendarific.com/holiday/india/pongal,india/pongal,All,All,2026-01-14 00:00:00+00:00
3,Makar Sankranti,Makar Sankranti is a restricted holiday in India,"{'id': 'in', 'name': 'India'}","{'iso': '2026-01-14', 'datetime': {'year': 202...","[Hinduism, Optional holiday]",Restricted Holiday,https://calendarific.com/holiday/india/makar-s...,india/makar-sankranti,All,All,2026-01-14 00:00:00+00:00
4,Vasant Panchami,"Vasant Panchami, or India's spring festival, i...","{'id': 'in', 'name': 'India'}","{'iso': '2026-01-23', 'datetime': {'year': 202...","[Hinduism, Optional holiday]",Restricted Holiday,https://calendarific.com/holiday/india/vasant-...,india/vasant-panchami,All,All,2026-01-23 00:00:00+00:00
5,Republic Day,India's Republic Day marks the anniversary of ...,"{'id': 'in', 'name': 'India'}","{'iso': '2026-01-26', 'datetime': {'year': 202...",[National holiday],Gazetted Holiday,https://calendarific.com/holiday/india/republi...,india/republic-day,All,All,2026-01-26 00:00:00+00:00
6,Guru Ravidas Jayanti,Guru Ravidas Jayanti is a restricted holiday i...,"{'id': 'in', 'name': 'India'}","{'iso': '2026-02-01', 'datetime': {'year': 202...",[Optional holiday],Restricted Holiday,https://calendarific.com/holiday/india/guru-ra...,india/guru-ravidas-jayanti,All,All,2026-02-01 00:00:00+00:00
7,Maharishi Dayanand Saraswati Jayanti,Maharishi Dayanand Saraswati Jayanti is a rest...,"{'id': 'in', 'name': 'India'}","{'iso': '2026-02-12', 'datetime': {'year': 202...",[Optional holiday],Restricted Holiday,https://calendarific.com/holiday/india/maharis...,india/maharishi-dayanand-saraswati-jayanti,All,All,2026-02-12 00:00:00+00:00
8,Valentine's Day,Valentine's Day is celebrated in many places w...,"{'id': 'in', 'name': 'India'}","{'iso': '2026-02-14', 'datetime': {'year': 202...",[Observance],Observance,https://calendarific.com/holiday/india/valenti...,india/valentine-day,All,All,2026-02-14 00:00:00+00:00
9,Maha Shivaratri,Maha Shivratri is an annual festival dedicated...,"{'id': 'in', 'name': 'India'}","{'iso': '2026-02-15', 'datetime': {'year': 202...","[Hinduism, Optional holiday]",Restricted Holiday,https://calendarific.com/holiday/india/maha-sh...,india/maha-shivaratri-shivaratri,All,All,2026-02-15 00:00:00+00:00


In [184]:
df['date_parsed'].dt.day_name()

0      Thursday
1      Saturday
2     Wednesday
3     Wednesday
4        Friday
        ...    
66       Monday
67    Wednesday
68     Thursday
69       Friday
70     Thursday
Name: date_parsed, Length: 71, dtype: str

In [4]:
def load_weather():

    return pd.read_json(
        "data/raw/weather/weather.json"
    )
df = load_weather()

In [7]:
meta_cell = df.loc[df["data"].notna(), "data"].iloc[0]
df = pd.DataFrame(meta_cell)

In [9]:
df["is_hot"] = (
        df["temp"] > 30
    )

In [10]:
df

,app_temp,aqi,city_name,clouds,country_code,datetime,dewpt,dhi,dni,elev_angle,...,timezone,ts,uv,vis,weather,wind_cdir,wind_cdir_full,wind_dir,wind_spd,is_hot
0,34.2,104,Āsansol,34,IN,2026-05-30:17,25.8,0,0,-44.66,...,Asia/Kolkata,1780164091,0,16,"{'code': 801, 'description': 'Few clouds', 'ic...",SSE,south-southeast,150,1.3,False


In [3]:
df = pd.read_parquet("data/processed/retail_features.parquet", engine="pyarrow")

In [4]:
df

,id,title,description,category,price,discountpercentage,rating,stock,tags,brand,...,returnpolicy,minimumorderquantity,meta,images,thumbnail,price_bucket,inventory_value,holiday_count,is_hot,is_humid
0,1,Essence Mascara Lash Princess,The Essence Mascara Lash Princess is a popular...,beauty,9.99,10.48,2.56,99,"[""beauty"", ""mascara""]",Essence,...,No return policy,48,"[[""barcode"", ""5784719087687""], [""createdAt"", ""...","[""https://cdn.dummyjson.com/product-images/bea...",https://cdn.dummyjson.com/product-images/beaut...,medium,989.01,71,False,True
1,2,Eyeshadow Palette with Mirror,The Eyeshadow Palette with Mirror offers a ver...,beauty,19.99,18.19,2.86,34,"[""beauty"", ""eyeshadow""]",Glamour Beauty,...,7 days return policy,20,"[[""barcode"", ""9170275171413""], [""createdAt"", ""...","[""https://cdn.dummyjson.com/product-images/bea...",https://cdn.dummyjson.com/product-images/beaut...,luxury,679.66,71,False,True
2,3,Powder Canister,The Powder Canister is a finely milled setting...,beauty,14.99,9.84,4.64,89,"[""beauty"", ""face powder""]",Velvet Touch,...,No return policy,22,"[[""barcode"", ""8418883906837""], [""createdAt"", ""...","[""https://cdn.dummyjson.com/product-images/bea...",https://cdn.dummyjson.com/product-images/beaut...,premium,1334.11,71,False,True
3,4,Red Lipstick,The Red Lipstick is a classic and bold choice ...,beauty,12.99,12.16,4.36,91,"[""beauty"", ""lipstick""]",Chic Cosmetics,...,7 days return policy,40,"[[""barcode"", ""9467746727219""], [""createdAt"", ""...","[""https://cdn.dummyjson.com/product-images/bea...",https://cdn.dummyjson.com/product-images/beaut...,premium,1182.09,71,False,True
4,5,Red Nail Polish,The Red Nail Polish offers a rich and glossy r...,beauty,8.99,11.44,4.32,79,"[""beauty"", ""nail polish""]",Nail Couture,...,No return policy,22,"[[""barcode"", ""4063010628104""], [""createdAt"", ""...","[""https://cdn.dummyjson.com/product-images/bea...",https://cdn.dummyjson.com/product-images/beaut...,medium,710.21,71,False,True
5,6,Calvin Klein CK One,CK One by Calvin Klein is a classic unisex fra...,fragrances,49.99,1.89,4.37,29,"[""fragrances"", ""perfumes""]",Calvin Klein,...,90 days return policy,9,"[[""barcode"", ""2451534060749""], [""createdAt"", ""...","[""https://cdn.dummyjson.com/product-images/fra...",https://cdn.dummyjson.com/product-images/fragr...,luxury,1449.71,71,False,True
6,7,Chanel Coco Noir Eau De,Coco Noir by Chanel is an elegant and mysterio...,fragrances,129.99,16.51,4.26,58,"[""fragrances"", ""perfumes""]",Chanel,...,No return policy,1,"[[""barcode"", ""4091737746820""], [""createdAt"", ""...","[""https://cdn.dummyjson.com/product-images/fra...",https://cdn.dummyjson.com/product-images/fragr...,luxury,7539.42,71,False,True
7,8,Dior J'adore,J'adore by Dior is a luxurious and floral frag...,fragrances,89.99,14.72,3.80,98,"[""fragrances"", ""perfumes""]",Dior,...,7 days return policy,10,"[[""barcode"", ""1445086097250""], [""createdAt"", ""...","[""https://cdn.dummyjson.com/product-images/fra...",https://cdn.dummyjson.com/product-images/fragr...,luxury,8819.02,71,False,True
8,9,Dolce Shine Eau de,Dolce Shine by Dolce & Gabbana is a vibrant an...,fragrances,69.99,0.62,3.96,4,"[""fragrances"", ""perfumes""]",Dolce & Gabbana,...,7 days return policy,2,"[[""barcode"", ""3023868210708""], [""createdAt"", ""...","[""https://cdn.dummyjson.com/product-images/fra...",https://cdn.dummyjson.com/product-images/fragr...,luxury,279.96,71,False,True
9,10,Gucci Bloom Eau de,Gucci Bloom by Gucci is a floral and captivati...,fragrances,79.99,14.39,2.74,91,"[""fragrances"", ""perfumes""]",Gucci,...,No return policy,2,"[[""barcode"", ""3170832177880""], [""createdAt"", ""...","[""https://cdn.dummyjson.com/product-images/fra...",https://cdn.dummyjson.com/product-images/fragr...,luxury,7279.09,71,False,True
